# Task 3: AQE Experiment (Queries 1, 6)

Compare the same Gold-table queries with Adaptive Query Execution (AQE) disabled and enabled.

In [ ]:
import sys

from pyspark.sql import SparkSession

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.benchmarking import measure_registered_query, rows_match
from analytics.runner import execute_registered_query, register_integrated_taxi_trips

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('Assignment2-AQE-Queries1-6')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')

QUERY_NAMES = (
    'query_01_monthly_zone_demand',
    'query_06_monthly_taxi_demand_trend',
)
REPETITIONS = 9
AQE_DISABLED = 'false'
AQE_ENABLED = 'true'

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/19 10:34:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Measure AQE disabled and enabled

Each mode receives one warm-up followed by three timed runs.

In [2]:
def measure_aqe_mode(enabled):
    spark.catalog.clearCache()
    spark.conf.set('spark.sql.adaptive.enabled', enabled)
    register_integrated_taxi_trips(spark)
    return {
        query_name: measure_registered_query(spark, query_name, REPETITIONS)
        for query_name in QUERY_NAMES
    }

aqe_disabled_results = measure_aqe_mode(AQE_DISABLED)
aqe_enabled_results = measure_aqe_mode(AQE_ENABLED)

for query_name in QUERY_NAMES:
    assert rows_match(
        aqe_disabled_results[query_name]['rows'],
        aqe_enabled_results[query_name]['rows'],
    ), f'AQE results differ for {query_name}.'

print('AQE-enabled and AQE-disabled results match within float tolerance.')

26/09/19 10:34:47 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/09/19 10:34:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


AQE-enabled and AQE-disabled results match within float tolerance.


## AQE-disabled physical plans

In [3]:
spark.conf.set('spark.sql.adaptive.enabled', AQE_DISABLED)
register_integrated_taxi_trips(spark)

for query_name in QUERY_NAMES:
    print(f'AQE-disabled plan: {query_name}')
    execute_registered_query(spark, query_name).explain('formatted')

AQE-disabled plan: query_01_monthly_zone_demand
== Physical Plan ==
* Sort (8)
+- Exchange (7)
   +- * HashAggregate (6)
      +- Exchange (5)
         +- * HashAggregate (4)
            +- * Project (3)
               +- * ColumnarToRow (2)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [pickup_date#2540, pickup_zone_name#2547, date#2581]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trips]
ReadSchema: struct<pickup_date:string,pickup_zone_name:string>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [pickup_date#2540, pickup_zone_name#2547, date#2581]

(3) Project [codegen id : 1]
Output [2]: [pickup_zone_name#2547, substring(pickup_date#2540, 1, 7) AS pickup_month#2582]
Input [3]: [pickup_date#2540, pickup_zone_name#2547, date#2581]

(4) HashAggregate [codegen id : 1]
Input [2]: [pickup_zone_name#2547, pickup_month#2582]
Keys [2]: [pickup_zone_name#2547, pickup_month#2582]
Functions [1]: [partial_count(1)]
Aggregate 

## AQE-enabled physical plans

AEQ enabled -> `AdaptiveSparkPlan` and changed shuffles or joins.

In [4]:
spark.conf.set('spark.sql.adaptive.enabled', AQE_ENABLED)
register_integrated_taxi_trips(spark)

for query_name in QUERY_NAMES:
    print(f'AQE-enabled plan: {query_name}')
    query = execute_registered_query(spark, query_name)
    query.collect()
    query.explain('formatted')

AQE-enabled plan: query_01_monthly_zone_demand
== Physical Plan ==
AdaptiveSparkPlan (20)
+- == Final Plan ==
   ResultQueryStage (13)
   +- * Sort (12)
      +- AQEShuffleRead (11)
         +- ShuffleQueryStage (10), Statistics(sizeInBytes=45.5 KiB, rowCount=790)
            +- Exchange (9)
               +- * HashAggregate (8)
                  +- AQEShuffleRead (7)
                     +- ShuffleQueryStage (6), Statistics(sizeInBytes=298.3 KiB, rowCount=5.19E+3)
                        +- Exchange (5)
                           +- * HashAggregate (4)
                              +- * Project (3)
                                 +- * ColumnarToRow (2)
                                    +- Scan parquet  (1)
+- == Initial Plan ==
   Sort (19)
   +- Exchange (18)
      +- HashAggregate (17)
         +- Exchange (16)
            +- HashAggregate (15)
               +- Project (14)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [pickup_date#2865, pickup_zone_name

## Benchmark summary

In [5]:
summary = []
for query_name in QUERY_NAMES:
    disabled = aqe_disabled_results[query_name]
    enabled = aqe_enabled_results[query_name]
    summary.append({
        'query': query_name,
        'aqe_disabled_samples_seconds': [
            round(value, 3) for value in disabled['samples_seconds']
        ],
        'aqe_disabled_median_seconds': round(disabled['median_seconds'], 3),
        'aqe_enabled_samples_seconds': [
            round(value, 3) for value in enabled['samples_seconds']
        ],
        'aqe_enabled_median_seconds': round(enabled['median_seconds'], 3),
        'aqe_speedup_factor': round(
            disabled['median_seconds'] / enabled['median_seconds'],
            2,
        ),
    })

summary

[{'query': 'query_01_monthly_zone_demand',
  'aqe_disabled_samples_seconds': [1.06, 0.809, 0.621],
  'aqe_disabled_median_seconds': 0.809,
  'aqe_enabled_samples_seconds': [0.534, 0.546, 0.508],
  'aqe_enabled_median_seconds': 0.534,
  'aqe_speedup_factor': 1.52},
 {'query': 'query_06_monthly_taxi_demand_trend',
  'aqe_disabled_samples_seconds': [0.492, 0.466, 0.443],
  'aqe_disabled_median_seconds': 0.466,
  'aqe_enabled_samples_seconds': [0.478, 0.383, 0.366],
  'aqe_enabled_median_seconds': 0.383,
  'aqe_speedup_factor': 1.22}]

In [6]:
spark.catalog.clearCache()
spark.stop()